# 📘 WooCommerce Data Pipeline (Source → Bronze → Silver → Gold)

**Version:** 1.1  
**Author:** Rita Pereira  
**Goal:** Document the analytical requirements and define the Gold tables and Silver → Gold transformations needed to produce datasets ready for consumption in BI tools such as Power BI or Tableau. The overall pipeline structure follows the medallion architecture (Source → Bronze → Silver → Gold). 

# 1. Overview
This repository implements a daily data pipeline for WooCommerce, built entirely in Python. It reads raw CSV exports from `SOURCE`, converts them to Parquet in `BRONZE`, applies cleaning and standardization in `SILVER`, and builds analytical models in `GOLD`.
 
**Architecture**: SOURCE → BRONZE (exact copy) → SILVER (cleaning & standardization) → GOLD (analytical model for BI)  

Data folders:

/source/YYYY/MM/DD/  
/bronze/YYYY/MM/DD/  
/silver/YYYY/MM/DD/  
/gold/YYYY/MM/DD/  

---

# 2. Input Data (SOURCE)
Place the WooCommerce CSV exports in:  
```
data/source/YYYY/MM/DD/
```

This project covers 4 source files from WooCommerce:
1. order_items.csv
2. order_itemmeta.csv
3. orders_with_staff_assignment.csv
4. post_with_hours.csv

**With auto-discovery enabled**, additional CSV files can be added without modifying any code.

---

# 3. Goal of the Gold Model (Business View)  

The goal of the Gold layer is to produce a **clean, reliable, and BI-ready model** that answers questions like:  

### Sales – Descriptive & Predictive

- How much have we sold per day / week / month?
- Which products generate the most revenue?
- Which sizes and colors sell the most?
- Which customers buy the most? (by total amount, frequency, margin…)
- Which customers have increased or decreased their spending lately?
- What is the sales forecast for the next week or month?
- Which products will see the highest demand in the coming days?
- Which products are showing a declining sales trend?
- On which days of the week is the highest sales volume expected?
- Which products are frequently bought together (basket analysis)?
- Which products show the strongest seasonality?

### Stores & Staff – Predictive Operations

- Which store sells the most?
- Which staff member handles the most orders?
- How are products distributed across stores?
- Which store is most likely to hit its monthly target?
- Which stores show a growth or decline pattern?
- Which staff members handle the most orders during peak times?
- Which stores need restocking sooner than others?
- How are sales distributed geographically across stores?
- During which time slots is the highest traffic or sales volume expected?

### Products – Mix, Turnover & Forecast

- Which sizes sell the most per category?
- Which colors are the most popular?
- Which virtual products sell the most?
- Which products have the highest turnover and which are stagnating?
- Which products are at risk of going out of stock based on sales trends?
- Which products are at risk of overstock?
- Which categories are likely to see higher volumes next month?
- Which size–color combinations are most likely to sell out?
- Which products have the highest margin per unit sold?

### eCommerce – Performance & Customer Behavior

- How are sales evolving over time?
- What is the average order value?
- Which orders are virtual vs. physical?
- What percentage of visitors end up buying (conversion rate)?
- Which customer segment buys more virtual vs. physical products?
- How will the average order value change in the coming weeks?
- What cart abandonment patterns exist?
- Which channels (email, social, organic…) drive the most sales?
- Which products attract the most traffic but have the lowest conversion?

### Customers – Profiling, Segmentation & Prediction

- Which customers are most likely to make a purchase soon?
- Which customers are at risk of churning?
- What is the customer lifetime value (LTV) per segment?
- Which customer segments buy more virtual products?
- Which customers respond best to discounts or promotions?
- How does purchasing behavior differ between new and returning customers?
- Which customer groups share similar purchasing patterns (clustering)?

### Logistics & Inventory – Operational Predictions

- Which products need restocking this week based on current trends?
- How much inventory will be needed to meet demand next month?
- Which products could run out of stock if no action is taken?
- Which store will have the greatest restocking need based on projected sales?

This clearly defines **which columns are mandatory in Gold** and which can be dropped.  

---

# 4. Pipeline Tools & Technology

This pipeline is built with a simple, reproducible, 100% Python approach, using accessible and stable tools suited to the current data scale of the project. Below is a summary of all tools used across ingestion, cleaning, transformation, and modelling.

## 4.1 Python + Pandas (core engine)

The pipeline is built around **Python** and **Pandas** as the main data processing library. Used for:

- reading CSV files (SOURCE)
- writing Parquet files (BRONZE)
- cleaning, normalization, and validation (SILVER)
- building dimensions and fact tables (GOLD)
- handling types, dates, strings, and joins

**Why Pandas:** Ideal for medium-sized datasets, notebook-based development, and reproducible pipelines without complex infrastructure.

## 4.2 Folder Structure (simple, auditable ETL)  
The pipeline uses daily date-partitioned folders:  
```
data/source/YYYY/MM/DD/
data/bronze/YYYY/MM/DD/
data/silver/YYYY/MM/DD/
data/gold/YYYY/MM/DD/
```

This enables:

- daily versioning
- full pipeline reproducibility
- data traceability
- easy debugging
- complete ingestion history


## 4.3 Python Standard Library  
Standard library modules used:

- os → path management
- glob → file auto-discovery
- pathlib → path-safe file management
- datetime → date handling
- json → metadata reading where needed

**Why:** Avoids unnecessary dependencies and keeps the pipeline portable and lightweight.

## 4.4 Jupyter Notebook  
Used as an interface for:

- running pipeline code step by step
- living documentation
- validating results at each stage
- data exploration
- prototyping the Gold model

## 4.5 Data Formats  

**CSV (SOURCE)**
- Native WooCommerce export format.

**Parquet (BRONZE / SILVER / GOLD)**  
Used for all intermediate and final storage:

- columnar format
- compressed
- fast read/write
- compatible with BI engines

**Why:** Optimizes both performance and disk space.

## 4.6 Pandas Cleaning & Transformation Methods  
Core Pandas methods used throughout the pipeline:

- `.strip()` and `.replace()` → string cleaning
- `.astype()` → type casting
- `.to_datetime()` → date parsing
- `.drop_duplicates()` → deduplication
- `.merge()` → joins
- `.pivot_table()` → key-value to columnar transformation
- `.apply()` → custom logic
- `.fillna()` → null handling
- `.query()` → semantic filtering


## 4.7 Pipeline Architecture  
The pipeline follows the classic medallion architecture **SOURCE → BRONZE → SILVER → GOLD**:

**SOURCE**  
Original CSV files as exported from WooCommerce. No modifications.

**BRONZE**  
Exact Parquet copy of the source files (no cleaning). Serves as a reproducible backup.

**SILVER**  
Cleaning, type casting, and normalization layer:

- consistent types
- clean strings
- parsed dates
- validations
- key-value pivoting
- dropping irrelevant columns

**GOLD**  
Final analytical model for BI:

- **dim_staff**
- **dim_time**
- **dim_order**
- **fact_order_items**
- **fact_orders_time**
- **fact_staff_orders**

**Why:**
Ensures data quality, scalability, and a clean star schema.

---

# 5. Silver & Gold Transformations  
## 5.1 Tables: order_items and order_itemmeta  

**Goal**: Standardize and enrich each order line to build the `fact_order_items` table (grain = 1 row per `order_item_id`) and link it to the star schema dimensions (`dim_order` via `order_id`, and a future `dim_product` via `product_id`/`variation_id` if built in a later phase).

### 5.1.1 order_items: Silver Transformations  

**Goal**: Normalize line item information (name, color, size) and ensure consistent types.

#### Transformations applied

1. **Type casting**  
    - `order_item_id` → `Int64`
    - `order_id` → `Int64`  
2. **Splitting the order_item_name field**
    - Using: `rsplit(" - ", 2)` (right-split, max 2 cuts)
    - Resulting columns:
        - `order_item_name`
        - `order_item_color` *(optional, may be NULL)*
        - `order_item_size` *(optional, may be NULL)*
    - Right-split avoids breaking product names that contain ` - ` internally
3. **Text cleaning and normalization**  
    - `strip()` on all text columns
    - Optional normalization (e.g. `title()` per business convention)
    - Collapse double spaces with `replace()`
4. **Dropping irrelevant columns**
    - `order_item_type` (always `"line_item"` — adds no value)

#### Silver Validations  

- `order_item_id` is unique 
- The split produces 1 to 3 parts without errors
- No unnecessary columns remain

**Code example**  
```python
# Type casting
order_items_silver["order_item_id"] = order_items_silver["order_item_id"].astype(int)
order_items_silver["order_id"] = order_items_silver["order_id"].astype(int)

# Right-split the name field (safe for names containing ' - ')
parts = order_items_silver["order_item_name"].str.rsplit(" - ", n=2, expand=True)
order_items_silver["order_item_name"]  = parts[0].str.strip()
order_items_silver["order_item_color"] = parts[1].str.strip() if parts.shape[1] > 1 else None
order_items_silver["order_item_size"]  = parts[2].str.strip() if parts.shape[1] > 2 else None

# Text cleanup
for col in ["order_item_name", "order_item_color", "order_item_size"]:
    order_items_silver[col] = order_items_silver[col].astype("string").str.strip()
```

### 5.1.2 order_itemmeta: Silver Transformations  

**Goal**: Convert WooCommerce's native *key-value* structure into analytical columns (`product_id`, `variation_id`, `quantity`, monetary amounts…) suitable for the star schema.

**Note: Native structure in SOURCE/BRONZE (unchanged):**

The `order_itemmeta` table arrives from WooCommerce with only 4 fixed columns, preserved as-is in BRONZE:

- meta_id
- order_item_id
- meta_key
- meta_value

This is WooCommerce's native format — a **key-value** structure where each row describes one attribute of an order line item.

### Silver Transformations

SILVER is where the main structural transformation happens.

#### Transformations applied

1. **Type casting**  
    - `meta_id` → `Int64`
    - `order_item_id` → `Int64`

2. **Text cleaning**
    - `.strip()` to remove leading/trailing spaces
    - `.replace()` to remove double spaces or unwanted symbols

3. **Filtering relevant keys**

Only analytically useful meta_keys are kept:

| meta_key | Columna final |
|-----------|-----------|
| _product_id | product_id|
| _variation_id | variation_id |
| _qty | quantity |
| _line_total | line_total |
| _line_tax | line_tax |
| _line_subtotal | line_subtotal |
| _line_subtotal_tax | line_subtotal_tax|

*All other WooCommerce meta keys are discarded in Silver.*

4. **Pivoting key → columns**
- From: `order_item_id | meta_key | meta_value`
- To: `order_item_id | product_id | variation_id | quantity | line_total | line_tax | …`

This converts the **key-value** format into a **columnar** format ready for analysis.

5. **Numeric normalization**
- `quantity` → `int`
- Spanish decimal format is handled before converting to float:
    - `"12,50"` → `12.50`
    - `"1.234,50"` → `1234.50`

6. **Dropping temporary columns**
    - `meta_key`, `meta_value` (and pivot auxiliaries)

#### Silver Validations  

- One row per `order_item_id` after pivoting
- Correct numeric conversion of monetary amounts
- `quantity >= 1` (or a defined policy for returns/refunds)
- `product_id` non-null in most rows (shipping, fees, and discount lines may have no `product_id` — this is expected WooCommerce behavior)

**Code example**

```python
# Filter to analytical keys only
keys = [
    "_product_id","_variation_id","_qty",
    "_line_total","_line_tax","_line_subtotal","_line_subtotal_tax"
]
meta_f = order_itemmeta_silver[order_itemmeta_silver["meta_key"].isin(keys)].copy()

# Pivot from long format to wide
meta_pivot = (
    meta_f
    .pivot_table(index="order_item_id", columns="meta_key", values="meta_value", aggfunc="last")
    .reset_index()
)

# Rename from WooCommerce internal keys to clean column names
meta_pivot = meta_pivot.rename(columns={
    "_product_id":"product_id",
    "_variation_id":"variation_id",
    "_qty":"quantity",
    "_line_total":"line_total",
    "_line_tax":"line_tax",
    "_line_subtotal":"line_subtotal",
    "_line_subtotal_tax":"line_subtotal_tax",
})

# Type casting
for c in ["product_id","variation_id","quantity"]:
    meta_pivot[c] = pd.to_numeric(meta_pivot[c], errors="coerce").astype("Int64")
for c in ["line_total","line_tax","line_subtotal","line_subtotal_tax"]:
    meta_pivot[c] = pd.to_numeric(meta_pivot[c], errors="coerce")
```

### 5.1.3 Gold – fact_order_items (final integration)  

**Goal**: Consolidate `order_items_silver` + `order_itemmeta_silver` into a **line-level fact table**, BI-ready and connected to the star schema.

**Join performed**

- 1:1 join on `order_item_id` (LEFT JOIN to keep all lines even if metadata is missing).
```python
fact_order_items = order_items_silver.merge(
    meta_pivot, on="order_item_id", how="left"
)
```

**Derived metrics**
```python
# unit_price only when quantity > 0
fact_order_items["unit_price"] = (
    fact_order_items.apply(
        lambda r: r["line_total"] / r["quantity"] if pd.notnull(r["line_total"]) and pd.notnull(r["quantity"]) and r["quantity"] > 0 else pd.NA,
        axis=1
    )
)

# total_with_tax (robusto)
# - If WooCommerce sends tax per line (`_line_tax`):
    total_with_tax = line_total + line_tax
# - If `_line_tax` is absent (tax included in price, as in this dataset):
    total_with_tax = line_total
```
#### Gold Relationships (star schema)

- `order_id` → FK to `dim_order.order_id`
- `product_id` → FK to a future `dim_product`
- `variation_id` → optional FK (only present if WooCommerce sends `_variation_id`)


**Important:** `fact_order_items` must not contain order-level attributes (store, date, status, order totals, etc.). All of that lives in `dim_order` and `dim_time`.

#### Final Gold structure – fact_order_items  

**Keys:**
- `order_item_id` *(fact PK at line level)*
- order_id
- product_id
- `variation_id` — optional, only if present in the metadata

**Product attributes (from order_items):**
- order_item_name
- order_item_color
- order_item_size

**Financial metrics (from order_itemmeta):**
- quantity
- line_total
- `line_tax` — optional; absent when prices already include tax
- `line_subtotal` — optional
- `line_subtotal_tax` — optional

**Derived metrics:**
- unit_price = line_total / quantity
- `total_with_tax` = `line_total` (+ `line_tax` if present)

**Technical columns:**
- file_date
- ingestion_timestamp
- processing_timestamp

#### Gold Validations

- Uniqueness: `(order_id, order_item_id)` combination is unique
- `quantity > 0` (or a clear policy for returns)
- monetary amounts `>= 0` where applicable
- `unit_price` computed only when `quantity > 0`
- `total_with_tax` equals `line_total` when `line_tax` is absent
- If `line_tax` is absent in the source (current case), this is by design — prices include VAT.
- Referential integrity:
    - `order_id` must exist in `dim_order` (audit for orphans)
    - `product_id` non-null in most rows

#### Data quality considerations

- Handle **returns/adjustments** if WooCommerce exposes them in these tables
- Normalize thousand separators and decimal commas in `meta_value` before casting to numeric
- Log **orphan orders** (items whose `order_id` is not in `dim_order`) for auditing
- WooCommerce may omit tax meta_keys (like `_line_tax`) when prices include VAT. In that case, `line_total` already includes tax — do not add any extra column.

---

## 5.2 Table: orders_with_staff_assignment

This table contains operational data about orders assigned to staff members, sourced from WooCommerce/WordPress. It feeds two Gold tables:

- **dim_staff** → staff dimension
- **fact_staff_orders** → order-staff assignment fact table

It also feeds the **dim_order** dimension because it contains key order attributes:

- order_id
- store_name
- num_items
- total_value
- `fecha` (converted to `order_timestamp`)

#### Characteristics of the original file  
The file contains:
- **3 redundant order ID columns** from the upstream join:
  `order_id_x`, `order_id_y`, `order_id`  
- WordPress/WooCommerce columns prefixed with `post_`
- Staff data:
    - staff_id
    - staff_name
- Order data:
    - store_name
    - num_items
    - total_value
    - `fecha` — order date/time as a string
- `is_virtual` almost always empty

### Silver Transformations

**Goal:** Standardize the data, keep only useful columns, and prepare the foundation for building stable dimensions and clean fact tables in Gold.

#### Transformations applied

1. **Selecting relevant columns**

From the 17 original fields, only these are kept:

| Column kept | Purpose |
| -----------------| -----|
| `order_id` | Order identifier |
| `store_name` | Store that handled the order |
| `staff_id` | Staff member identifier |
| `staff_name` | Staff member name |
| `num_items` | Total products in the order |
| `total_value` | Total order amount |
| `fecha` | Order date as string |
| `assignment_timestamp` | Date/time the order was assigned to staff

Columns dropped:  
- ID
- post_author
- post_title
- post_content
- post_status
- post_type
- order_id_x
- order_id_y
- is_virtual  
  *(no analytical value or redundant)*

2. **Normalizing the order identifier**

The source has 3 inconsistent `order_id` variants from the upstream join:
- order_id_x
- order_id_y
- order_id

In Silver, we validate they match and keep only `order_id`:

```python
assert df['order_id_x'].equals(df['order_id'])
assert df['order_id_y'].equals(df['order_id'])
```


3. **Type casting**

- `order_id` → `Int64`
- `staff_id` → `Int64`
- `num_items` → `Int64`
- `total_value` → `float`
- `fecha` → `datetime`
- `assignment_timestamp` → `datetime` (from `post_date`)

```python
df['assignment_timestamp'] = pd.to_datetime(df['post_date'], errors='coerce')
df['fecha'] = pd.to_datetime(df['fecha'])
```

4. **String cleaning and normalization**

Applied to:  
- store_name
- staff_name

Includes:  
- .strip()
- fix capitalization (`title()` or business standard)
- collapse double spaces

5. **Additional normalized columns**
Aliases generated for analytical clarity:
- `assigned_staff_id`
- `assigned_staff_name`
- `assigned_store_name`

#### Silver Validations  
- `order_id` is not null
- `staff_id` is not null
- `assignment_timestamp` is valid
- `total_value >= 0`
- No exact duplicates on `(order_id, staff_id, assignment_timestamp)`

### Gold Transformations

This table produces three Gold artifacts:  
1. **dim_staff** → staff dimension
2. **fact_staff_orders** → order-staff fact table
3. **dim_order** → order dimension 

### 5.2.1 Gold – dim_staff (Staff Dimension)

**Goal**: Build a **stable, consistent** dimension representing staff members, independent of daily assignments. 

**Gold columns**

- `staff_id` (PK)
- staff_name
- staff_name_clean
- `store_name` (if the staff member always belongs to one store)
- `first_seen_date` (earliest assignment date for this staff member)
- `last_seen_date` (most recent assignment date)

#### Key transformations  
1. Select unique staff members:

```python
dim_staff = df_silver[['staff_id', 'staff_name']].drop_duplicates()
```

2. Clean the name:

```python
dim_staff['staff_name_clean'] = dim_staff['staff_name'].str.title()
```

3. Activity date range (min/max assignment timestamp):

```python
dim_staff['first_seen_date'] = df.groupby('staff_id')['assignment_timestamp'].min()
dim_staff['last_seen_date']  = df.groupby('staff_id')['assignment_timestamp'].max()
```

#### Validations:  
- `staff_id` is unique
- dates are valid
- `staff_name_clean` is not empty


### 5.2.2 Gold – fact_staff_orders (Order-Staff Fact Table)

**Goal**: Record each order-to-staff assignment, including:

- who handled it
- when
- volume handled
- order value

Grain: 1 row = 1 order assigned to 1 staff member.

**Gold columns**  

| Column | Purpose |
| ----- | -----|
| `order_id` | FK → dim_order |
| `staff_id` | FK → dim_staff |
| `assignment_timestamp` | when the order was assigned |
| `assignment_time_id` | FK → dim_time |
| `num_items` | number of items in the order |
| `total_value` | order total amount |
| `orders_handled` | base metric (always 1) |

Building `assignment_time_id`:

```python
df_gold['assignment_time_id'] = (
    df_gold['assignment_timestamp'].dt.year  * 1000000 +
    df_gold['assignment_timestamp'].dt.month * 10000 +
    df_gold['assignment_timestamp'].dt.day   * 100 +
    df_gold['assignment_timestamp'].dt.hour
)
```

#### Gold Validations  

- `(order_id, staff_id)` is unique
- `assignment_timestamp` is not null
- `total_value >= 0`
- `num_items >= 1`
- `staff_id` exists in `dim_staff`
- `order_id` exists in `dim_order`

### 5.2.3 Gold – dim_order (Conformed Order Dimension)

This dimension captures all structural order attributes:

- order timestamp (`time_id`)
- store (`store_name`)
- volume (`num_items`)
- total value (`total_value`)

**dim_order** is the **central table** of the star schema — all three fact tables link to it via `order_id`. More detail in section 5.4.

#### Final Gold output (from orders_with_staff_assignment)  

| Table | Type | Content |
| --- | --- | --- |
| `dim_staff` | Dimension | Staff information |
| `fact_staff_orders` | Fact | Order-staff assignments |
| `dim_order` | Conformed dimension | Order attributes (store, date, value…)

---

## 5.3 Table: post_with_hours → dim_time + fact_orders_time  

**Goal:** Build the time dimension and the order timestamp fact table from `post_with_hours`, resulting in a clean star schema:  
- `dim_time` → all temporal attributes (day, month, weekday name, hour bucket, weekend flag…)
- `fact_orders_time` → only keys and the order timestamp:
    - `order_id` (FK → dim_order)
    - `time_id` (FK → dim_time)
    - `order_timestamp` (for auditing)

### Silver source:  

Relevant columns:  
- `ID` → renamed to `order_id`  
- `post_date` → renamed to `order_timestamp`  
- `post_type` → filtered to `shop_order` only  
- `fecha` → string, not used here  

### Silver Transformations  

1. **Filtering**: `post_type == "shop_order"` (WordPress contains all post types mixed)

2. **Type conversions**  
  
```python
df['order_timestamp'] = pd.to_datetime(df['post_date'], errors='coerce')
df['order_id'] = df['ID'].astype(int)
```

3. **Validations**

- `order_timestamp` is not null
- `order_id` is unique
- Values are within expected ranges (hours 0–23, months 1–12)

### 5.3.1 Gold – dim_time (Time Dimension)

**Gold columns**  

- time_id (YYYYMMDDHH)
- order_timestamp
- order_date
- order_year
- order_month
- order_month_name
- order_day
- order_hour
- `order_hour_bucket` (morning / afternoon / night)
- `order_weekday` (Monday = 0)
- order_weekday_name
- `is_weekend` (True for Saturday/Sunday)

**Python transformations**
 
```python
dim_src = post_with_hours_silver[["post_date"]].drop_duplicates().rename(columns={"post_date": "order_timestamp"})
dim_src["order_timestamp"] = pd.to_datetime(dim_src["order_timestamp"])

dim_time = dim_src.copy()
dim_time["order_date"]          = dim_time["order_timestamp"].dt.date
dim_time["order_year"]          = dim_time["order_timestamp"].dt.year
dim_time["order_month"]         = dim_time["order_timestamp"].dt.month
dim_time["order_month_name"]    = dim_time["order_timestamp"].dt.strftime("%B")
dim_time["order_day"]           = dim_time["order_timestamp"].dt.day
dim_time["order_hour"]          = dim_time["order_timestamp"].dt.hour
dim_time["order_weekday"]       = dim_time["order_timestamp"].dt.weekday
dim_time["order_weekday_name"]  = dim_time["order_timestamp"].dt.strftime("%A")

def bucket(h):
    if 6 <= h < 13: return "morning"
    if 13 <= h < 20: return "afternoon"
    return "night"

dim_time["order_hour_bucket"] = dim_time["order_hour"].apply(bucket)
dim_time["is_weekend"]        = dim_time["order_weekday"].apply(lambda x: x >= 5)

dim_time["time_id"] = (
    dim_time["order_year"]  * 1000000 +
    dim_time["order_month"] * 10000   +
    dim_time["order_day"]   * 100     +
    dim_time["order_hour"]
)

dim_time = dim_time[[
    "time_id","order_timestamp","order_date","order_year","order_month","order_month_name",
    "order_day","order_hour","order_hour_bucket","order_weekday","order_weekday_name","is_weekend"
]].drop_duplicates(subset=["time_id"])
```

#### Validations:  
- `time_id` is unique
- `order_timestamp` is valid
- derived columns are consistent (month 1–12, hour 0–23, etc.)

### 5.3.2 Gold – fact_orders_time (Order-Time Fact Table)

**Gold columns**   
- `order_id` (FK → dim_order)
- `time_id` (FK → dim_time)
- order_timestamp

**Python transformations:**  
```python
fact_orders_time = (
    post_with_hours_silver
    .query("post_type == 'shop_order'")
    .rename(columns={"ID":"order_id","post_date":"order_timestamp"})
    [["order_id","order_timestamp"]]
    .copy()
)
fact_orders_time["order_timestamp"] = pd.to_datetime(fact_orders_time["order_timestamp"])

fact_orders_time["time_id"] = (
    fact_orders_time["order_timestamp"].dt.year  * 1000000 +
    fact_orders_time["order_timestamp"].dt.month * 10000   +
    fact_orders_time["order_timestamp"].dt.day   * 100     +
    fact_orders_time["order_timestamp"].dt.hour
)
```

#### Gold Validations

- `order_id` is unique
- `order_timestamp` is valid
- **Referential integrity:**
    - `time_id` exists in `dim_time`
    - `order_id` exists in `dim_order` (audit orphans with `isin()`)

#### Final Gold result

- `dim_time` holds **all temporal semantics**
- `fact_orders_time` is **narrow** and aligned to `dim_order` and `dim_time`
- **No duplicated time columns** in the fact table

---

## 5.4 Gold – dim_order (Conformed Order Dimension)

**Goal:** Build the **order dimension** with **one row per `order_id`**, consolidating the order's own attributes and serving as the central node connecting all fact tables:  
- fact_order_items
- fact_orders_time
- fact_staff_orders

**Inputs**
- `orders_with_staff_assignment_silver` (primary)
    - `order_id`, `store_name`, `num_items`, `total_value`, `fecha` → `order_timestamp`
- `post_with_hours_silver` → only if needed to fill in missing timestamps

#### Gold Transformations

1. **Timestamps**

```python
df = orders_with_staff_assignment_silver.copy()
df["order_timestamp"] = pd.to_datetime(df["fecha"], errors="coerce")
df["order_date"]      = df["order_timestamp"].dt.date
```

2. **Dropping temporal derived columns**

All columns like the following are excluded from `dim_order` because they belong exclusively to `dim_time`:

- order_year
- order_month
- order_month_name
- order_day
- order_hour
- order_weekday
- order_weekday_name
- order_hour_bucket
- is_weekend



3. **time_id (format: YYYYMMDDHH)**

```python
df["time_id"] = (
    df["order_timestamp"].dt.year   * 1000000 +
    df["order_timestamp"].dt.month  * 10000   +
    df["order_timestamp"].dt.day    * 100     +
    df["order_timestamp"].dt.hour
)
```

4. **Final column selection and deduplication by order**

```python
dim_order = df[[
    "order_id",
    "order_timestamp",
    "order_date",
    "time_id",
    "store_name",
    "num_items",
    "total_value"
]].drop_duplicates(subset=["order_id"]) \
  .sort_values("order_id") \
  .reset_index(drop=True)
```

#### Final structure – dim_order

- `order_id` (PK)
- order_timestamp
- order_date
- `time_id` (FK → dim_time)
- store_name
- num_items
- total_value

**Validations**

- `order_id` is unique
- `order_timestamp` is valid
- `num_items >= 0`, `total_value >= 0`
- `time_id` is consistent with `order_timestamp`
- 1:1 consistency — one order maps to exactly one `time_id`

---

# 6. Star Schema (ERD)

**Legend**
Keys:
- PK = Primary Key
- FK = Foreign Key

Naming convention:
- Fact tables: `fact_*`
- Dimension tables: `dim_*`

#### Star schema relationships

- fact_order_items.order_id → dim_order.order_id
- fact_orders_time.order_id → dim_order.order_id
- fact_staff_orders.order_id → dim_order.order_id
- fact_orders_time.time_id → dim_time.time_id
- fact_staff_orders.assignment_time_id → dim_time.time_id
- fact_staff_orders.staff_id → dim_staff.staff_id

```mermaid


erDiagram

    dim_staff {
        int staff_id PK
        string staff_name
        string staff_name_clean
        string store_name
        date first_seen_date
        date last_seen_date
    }

    dim_time {
        int time_id PK
        datetime order_timestamp
        date order_date
        int order_year
        int order_month
        string order_month_name
        int order_day
        int order_hour
        string order_hour_bucket
        int order_weekday
        string order_weekday_name
        boolean is_weekend
    }

    dim_order {
        int order_id PK        
        datetime order_timestamp
        date order_date
        int time_id FK
        string store_name
        int num_items
        float total_value
    }

    fact_staff_orders {
        int order_id FK
        int staff_id FK
        datetime assignment_timestamp
        int assignment_time_id FK
        int num_items
        float total_value
        int orders_handled
    }

    fact_order_items {
        int order_item_id PK
        int order_id FK
        int product_id FK
        string order_item_name
        string order_item_color
        string order_item_size
        int quantity
        float line_total
        float unit_price
        float total_with_tax
    }

    fact_orders_time {
        int order_id FK
        datetime order_timestamp
        int time_id FK
    }

    dim_staff ||--o{ fact_staff_orders : assigned_to
    dim_time  ||--o{ fact_orders_time : order_time
    dim_time  ||--o{ fact_staff_orders : assignment_time
    dim_order ||--o{ fact_orders_time : order_info
    dim_order ||--o{ fact_staff_orders : order_info
    dim_order ||--o{ fact_order_items : order_info


# 7. Full Pipeline Documentation (End-to-End View)
This section summarizes the complete end-to-end flow, from raw data in SOURCE to the analytical Gold model ready for BI. Rather than repeating the detailed transformations from section 5, it presents the **architecture**, **table flow**, and **design decisions** that ensure consistency, traceability, and data quality.

## 7.1 Pipeline Architecture  
The pipeline follows a classic data engineering approach:

```
SOURCE → BRONZE → SILVER → GOLD → BI STAR SCHEMA
```

Each layer has a distinct purpose:

### SOURCE
Original CSV files exported from WooCommerce. No modifications.

### BRONZE
Exact Parquet copy of SOURCE. Preserves:
- original structure
- original errors
- original row order

**Goal** → maximum traceability and reproducibility.

### SILVER

**Cleaning, type casting, and normalization** layer:
- correct types (int, float, datetime)
- string cleaning (`strip()`, `.replace()`)
- normalized numeric values
- key-value pivoting
- dropping irrelevant columns
- timestamp derivation
- per-dataset validations

**Goal** → consistent, clean, and audited data.

### GOLD

Final analytical layer → **star schema**:

- **Dimensions:**
    - dim_staff
    - dim_time
    - dim_order  

- **Fact tables:**
    - fact_order_items
    - fact_orders_time
    - fact_staff_orders  

**Goal** → a clean, fast model ready for BI.

## 7.2 Full Table Map by Layer

**SOURCE / BRONZE**
- order_items.csv
- order_itemmeta.csv
- orders_with_staff_assignment.csv
- post_with_hours.csv

**SILVER**

| Source | Silver output | Purpose | 
| --- | --- | ---|
| order_items | order_items_silver | Clean order line items | 
| order_itemmeta | order_itemmeta_silver | Pivoted metadata | 
| orders_with_staff_assignment | orders_with_staff_assignment_silver | Clean orders + staff data
| post_with_hours | post_with_hours_silver | Clean timestamps, filtered to shop_order |

### GOLD (analytical model)

**Dimensions**
- `dim_staff` → staff members
- `dim_time` → hourly calendar
- `dim_order` → conformed order dimension (central hub of the model)

**Fact tables**
- `fact_order_items` → order line items
- `fact_orders_time` → order timestamps
- `fact_staff_orders` → order-to-staff assignments

## 7.3 Full Pipeline Flow (Diagram)

```mermaid
flowchart TD

    subgraph SOURCE
        S1(order_items)
        S2(order_itemmeta)
        S3(orders_with_staff_assignment)
        S4(post_with_hours)
    end

    subgraph BRONZE
        B1(order_items)
        B2(order_itemmeta)
        B3(orders_with_staff_assignment)
        B4(post_with_hours)
    end

    subgraph SILVER
        SI1(order_items_silver)
        SI2(order_itemmeta_silver)
        SI3(orders_with_staff_assignment_silver)
        SI4(post_with_hours_silver)
    end

    subgraph GOLD
        D1(dim_order)
        D2(dim_time)
        D3(dim_staff)
        F1(fact_order_items)
        F2(fact_orders_time)
        F3(fact_staff_orders)
    end

    S1 --> B1 --> SI1 --> F1
    S2 --> B2 --> SI2 --> F1
    S3 --> B3 --> SI3 --> F3 --> D3
    S4 --> B4 --> SI4 --> F2 --> D2
    
    F1 --> D1
    F2 --> D1
    F3 --> D1
```

`dim_order` acts as the central dimension connecting all fact tables via `order_id`.

## 7.4 Final Star Schema (Conceptual Summary)

### Dimensions
- `dim_order` (conformed): order
- `dim_time`: calendar
- `dim_staff`: staff members

### Fact tables
- `fact_order_items` → order line detail
- `fact_orders_time` → when the order happened
- `fact_staff_orders` → who handled it

### Keys
- `order_id` → links all fact tables to `dim_order`
- `time_id` → links time-based facts to `dim_time`
- `staff_id` → links `fact_staff_orders` to `dim_staff`


## 7.5 Referential Integrity

Before loading Gold, the following is verified:  

#### Horizontal integrity (facts → dims)
- All `order_id` values in:
    - fact_order_items
    - fact_orders_time
    - fact_staff_orders  
    **must exist in `dim_order`**

- All `time_id` values in:
    - fact_orders_time
    - fact_staff_orders  
    **must exist in `dim_time`**

- All `staff_id` values in `fact_staff_orders`
**must exist in `dim_staff`**

#### Orphan audit  

Orphan rows are detected with:  
```python
fact_order_items[~fact_order_items["order_id"].isin(dim_order["order_id"])]
```

If any appear → possible issue in the SOURCE data or in synchronization between datasets.

## 7.6 Final Pipeline Checklist

#### SOURCE

- [ ] All CSVs present
- [ ] Correct encoding
- [ ] Dates within expected range

#### BRONZE

- [ ] Exact copy of source files
- [ ] No unexpected modifications

#### SILVER

- [ ] Correct types (int, float, datetime)
- [ ] Clean columns (strip, replace)
- [ ] Correct pivot in order_itemmeta
- [ ] No duplicates
- [ ] Validations run

#### GOLD

- [ ] All dimensions created
- [ ] All fact tables created
- [ ] Referential integrity OK
- [ ] Correct grain per table
- [ ] Star schema is consistent
- [ ] No orphan fact rows
- [ ] No dimension attribute duplication in facts


## 7.7 Final Summary

The pipeline implements a solid, auditable architecture built on Python and Pandas, covering the SOURCE–BRONZE–SILVER–GOLD layers and a final star schema with three dimensions (order, time, staff) and three fact tables (order_items, orders_time, staff_orders).

The result is a clean, consistent, BI-ready model that enables analysis of orders, sales, line items, time patterns, and staff productivity — with full traceability at every step.

---

# Table of Contents

## 1. Overview
- Pipeline purpose
- General architecture: SOURCE → BRONZE → SILVER → GOLD

## 2. Input Data (SOURCE)
- Original WooCommerce CSV files
- Folder structure by date (YYYY/MM/DD)
- Description of the 4 source datasets

## 3. Gold Model Goal (Business View)
- Sales
- Stores & Staff
- Products
- eCommerce
- Customers
- Logistics & Inventory

## 4. Pipeline Tools
- Python + Pandas
- Folder structure (daily ETL)
- Python standard library
- Jupyter Notebook / VS Code
- Data formats: CSV and Parquet
- Main cleaning and transformation methods
- Layered architecture

## 5. Silver & Gold Transformations

### 5.1 Processing order_items + order_itemmeta

#### Silver:
- Cleaning, type casting, normalization
- Safe split of order_item_name
- Key-value pivot of order_itemmeta
- Validations

#### Gold:
- Building fact_order_items
- Financial metrics
- Referential integrity with dim_order

### 5.2 Processing orders_with_staff_assignment
#### Silver:

- Column selection
- order_id normalization
- Staff / store cleaning
- Creating assignment_timestamp

#### Gold:

- dim_staff (staff dimension)
- fact_staff_orders (assignment fact table)

### 5.3 Processing post_with_hours

#### Silver:

- Filtering to shop_order
- Timestamp cleaning
- Temporal attributes

#### Gold:

- dim_time (full time dimension)
- fact_orders_time (order timestamp fact table)

### 5.4 Building dim_order (conformed order dimension)

- Integration from orders_with_staff_assignment
- Order attributes
- time_id and relationship to dim_time
- Role within the star schema

## 6. Star Schema (ERD)

- Conceptual description
- Mermaid ER diagram
- Relationships between dimensions and facts
- Grain explanation per table

## 7. Full Pipeline Documentation
### 7.1 Architecture: SOURCE → BRONZE → SILVER → GOLD
### 7.2 Full table map
### 7.3 Pipeline flow diagram
### 7.4 Star schema summary
### 7.5 Referential integrity (orphan detection)
### 7.6 Final pipeline checklist
### 7.7 Executive summary

---
